# 2048 - MEC (Expectimax/Minimax)
Notebook principal para la experimentación y el registro de resultados.


## Parte 1: técnicas (Expectimax y Minimax + α-β)
- **Expectimax**: el entorno agrega una ficha (2/4) con probabilidades; el agente maximiza el valor esperado.
- **Minimax**: modela la inserción de fichas como un adversario; se implementa con **poda α-β** para reducir tiempo.
- Ambos usan `heuristics.evaluate(board, weights)` para evaluar estados hoja.

Comando de demo interactiva (opcional):
```bash
poetry run python Main.py
```


In [ ]:
from datetime import datetime
from GameBoard import GameBoard
from Agent import Agent
from Expectimax_Agent import ExpectimaxAgent
from Minimax_AlphaBeta_Agent import MinimaxAlphaBetaAgent
SMOOTH_HEAVY = {
    "empty": 280,
    "monotonicity": 2.2,
    "corner": 25,
    "smoothness": 4.0,
    "merges": 50,
}

int_to_string = ['UP', 'DOWN', 'LEFT', 'RIGHT']

def check_win(board: GameBoard):
    return board.get_max_tile() >= 2048

def play_once(agent: Agent, render: bool = True):
    board = GameBoard()
    done = False
    moves = 0
    if render:
        board.render()
    start = datetime.now()
    while not done:
        action = agent.play(board)
        if render:
            print(f"Next Action: {int_to_string[action]}  (move {moves})")
        done = board.play(action)
        done = done or check_win(board)
        if render:
            board.render()
        moves += 1
    duration = datetime.now() - start
    return {
        "moves": moves,
        "max_tile": board.get_max_tile(),
        "win": check_win(board),
        "duration": duration,
    }


### Partida rápida (sanity check)
Corre una partida sin render (rápido) para verificar que el agente juega y termina.


In [ ]:
agent = ExpectimaxAgent(depth=3, weights=SMOOTH_HEAVY)
result = play_once(agent, render=False)
print(result)


## Parte 2: funciones de evaluación (MEC)
Las heurísticas están en `heuristics.py` y combinan varias señales para que Expectimax/Minimax prefieran tableros jugables. Se evalúan en log2 para que las potencias de 2 sean lineales. Componentes usados en `evaluate`:
- Vacías (`count_empty`): libera espacio para no quedar bloqueado.
- Monotonicidad (`monotonicity`): filas/columnas con orden creciente o decreciente consistente.
- Smoothness (`smoothness`): penaliza saltos grandes entre adyacentes (valor negativo).
- Esquina (`max_tile_in_corner`): recompensa si la ficha máxima queda en una esquina.
- Merges posibles (`merges_possible`): pares adyacentes listos para combinar.
- Peso posicional serpiente (`positional_weight`): layout deseado con fichas grandes agrupadas en una esquina.

Los pesos por defecto (`DEFAULT_WEIGHTS`) priorizan vacías + merges + esquina; `bench.py` permite sobreescribirlos por CLI o usar presets.

### Presets de pesos (bench.py)
Disponibles en `bench.py` para combinar heurísticas sin tocar código:
- `baseline`, `tuned`, `corner_heavy`, `smooth_heavy`, `snake` (activa `positional`).
Podés pasar `--preset` o `--weights empty=300,monotonicity=2,...` (tiene prioridad sobre preset).
Para reproducir resultados entre corridas, usar `--seed`.


### Evidencia de combinaciones/ponderaciones
Para cumplir “distintas combinaciones y ponderadas de distintas formas” usamos presets (misma heurística, pesos distintos) y registramos métricas con `bench.py` + CSV + `summarize_results.py`.


In [ ]:
# Ejemplo de sensibilidad de heurísticas en un tablero fijo
import numpy as np
from heuristics import (
    evaluate,
    count_empty, monotonicity, smoothness, max_tile_in_corner, merges_possible, positional_weight,
)

sample_grid = np.array([
    [2, 4, 8, 16],
    [32, 64, 128, 256],
    [2, 0, 0, 0],
    [0, 0, 0, 0],
], dtype=float)

weights_sets = {
    "baseline": {"empty": 250, "monotonicity": 1.5, "corner": 25, "smoothness": 3.0, "merges": 50},
    "smooth_heavy": {"empty": 280, "monotonicity": 2.2, "corner": 25, "smoothness": 4.0, "merges": 50},
    "snake": {"empty": 320, "monotonicity": 2.0, "corner": 40, "smoothness": 2.5, "merges": 70, "positional": 0.5},
}

components = {
    "empty": count_empty(sample_grid),
    "monotonicity": monotonicity(sample_grid),
    "smoothness": smoothness(sample_grid),
    "corner": max_tile_in_corner(sample_grid),
    "merges": merges_possible(sample_grid),
    "positional": positional_weight(sample_grid),
}
print("Componentes en sample_grid:")
for k, v in components.items():
    print(f"  {k}: {v}")

print("\nScores por preset:")
for name, w in weights_sets.items():
    print(f"  {name}: {evaluate(sample_grid, w):.2f}")


## Parte 3: experimentación, pruebas y registro
Definición de pruebas (MEC):
- Comparar **presets** manteniendo agente/depth constantes (impacto de ponderaciones).
- Comparar **Minimax con poda vs sin poda** (impacto en tiempo).
- Métricas por episodio: `win`, `max_tile`, `moves`, `duration_sec`, `grid_sum` (CSV por corrida).
- Repeticiones: usamos hasta **10 episodios** por corrida final.


### Corridas preliminares (ya hechas)
Estas corridas cortas sirvieron para validar funcionamiento y comparar poda (no es necesario repetirlas si vas a usar las corridas finales):
- `expectimax_smooth_heavy_d3.csv`
- `minimax_smooth_heavy_d4_prune.csv`
- `minimax_smooth_heavy_d4_noprune.csv`


### Corridas finales (para informe, máximo 10 episodios)
Guardamos todo en `results_final/` y usamos `--seed 0` para reproducibilidad.

**Expectimax (depth=3):** comparar presets con 10 episodios.
**Minimax (depth=4):** comparar presets con poda (10 episodios).
**Impacto α-β:** comparar `smooth_heavy` con/sin poda (recomendado 3–5 episodios sin poda por tiempo).
**Baseline random (opcional):** 10 episodios para tener un baseline trivial.


### Ejecutar corridas finales
Ejecutar las siguientes celdas en orden. Esto genera los CSV en `results_final/` y deja una tabla en `results_final/summary.txt`.


In [ ]:
from pathlib import Path
Path("results_final").mkdir(exist_ok=True)
print("results_final/ listo")


In [ ]:
# Expectimax d3: presets (10 episodios)
!poetry run python bench.py --agent expectimax --depth 3 --episodes 10 --preset baseline     --seed 0 --output results_final/expectimax_baseline_d3_e10_seed0.csv
!poetry run python bench.py --agent expectimax --depth 3 --episodes 10 --preset tuned        --seed 0 --output results_final/expectimax_tuned_d3_e10_seed0.csv
!poetry run python bench.py --agent expectimax --depth 3 --episodes 10 --preset corner_heavy --seed 0 --output results_final/expectimax_corner_heavy_d3_e10_seed0.csv
!poetry run python bench.py --agent expectimax --depth 3 --episodes 10 --preset smooth_heavy --seed 0 --output results_final/expectimax_smooth_heavy_d3_e10_seed0.csv
!poetry run python bench.py --agent expectimax --depth 3 --episodes 10 --preset snake        --seed 0 --output results_final/expectimax_snake_d3_e10_seed0.csv


In [ ]:
# Minimax d4 con poda α-β: presets (10 episodios)
!poetry run python bench.py --agent minimax --depth 4 --episodes 10 --preset baseline     --seed 0 --output results_final/minimax_baseline_d4_prune_e10_seed0.csv
!poetry run python bench.py --agent minimax --depth 4 --episodes 10 --preset tuned        --seed 0 --output results_final/minimax_tuned_d4_prune_e10_seed0.csv
!poetry run python bench.py --agent minimax --depth 4 --episodes 10 --preset corner_heavy --seed 0 --output results_final/minimax_corner_heavy_d4_prune_e10_seed0.csv
!poetry run python bench.py --agent minimax --depth 4 --episodes 10 --preset smooth_heavy --seed 0 --output results_final/minimax_smooth_heavy_d4_prune_e10_seed0.csv
!poetry run python bench.py --agent minimax --depth 4 --episodes 10 --preset snake        --seed 0 --output results_final/minimax_snake_d4_prune_e10_seed0.csv


In [ ]:
# Impacto de poda (sin poda es muy lento; 3–5 episodios alcanza para evidenciar el impacto)
!poetry run python bench.py --agent minimax --depth 4 --episodes 5 --preset smooth_heavy --seed 0 --output results_final/minimax_smooth_heavy_d4_prune_e5_seed0.csv
!poetry run python bench.py --agent minimax --depth 4 --episodes 3 --preset smooth_heavy --seed 0 --no-pruning --output results_final/minimax_smooth_heavy_d4_noprune_e3_seed0.csv


In [ ]:
# Baseline RandomAgent (opcional)
!poetry run python bench.py --agent random --episodes 10 --seed 0 --output results_final/random_na_d0_e10_seed0.csv


### Consolidar resultados
Genera e imprime la tabla resumen y la guarda en `results_final/summary.txt`.


In [ ]:
import subprocess
from pathlib import Path

out = subprocess.check_output([
    "poetry", "run", "python", "summarize_results.py", "--pattern", "results_final/*.csv"
], text=True)
Path("results_final/summary.txt").write_text(out, encoding="utf-8")
print(out)
